# 02. Traditional ML Baselines — Quora Question Pairs
**Team 5 | Taehun Kim**

> **전제**: `01_preprocessing.ipynb`를 먼저 실행해 `outputs/` 폴더가 생성되어 있어야 합니다.

### 학습 모델
| # | 모델 | 피처 |
|---|------|------|
| 1 | Logistic Regression | 수동 피처 |
| 2 | Logistic Regression | TF-IDF |
| 3 | Linear SVM | TF-IDF |
| 4 | Naive Bayes | TF-IDF |
| 5 | Decision Tree | 수동 피처 |
| 6 | Random Forest | 수동 피처 |
| 7 | Gradient Boosting | 수동 피처 |

### 평가 지표
Accuracy, F1, Log Loss, Inference Time, Model Size

## 0. 환경 설정 & 데이터 로딩

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=False)

import os

# ─────────────────────────────────────────────────────────────────
# 경로 설정
#   Kaggle: 01_preprocessing의 출력을 Dataset으로 추가한 경우
#           Add Data → Your Work → 01_preprocessing 선택
#           → 파일이 /kaggle/input/<notebook-slug>/ 에 마운트됨
#   로컬:   01_preprocessing을 같은 폴더에서 실행한 경우
#           → outputs/ 폴더가 현재 디렉토리에 있음
# ─────────────────────────────────────────────────────────────────
KAGGLE_PREPROC = '/kaggle/input/team5-quora-preprocessed'  # ← Kaggle dataset 이름에 맞게 수정
LOCAL_PREPROC  = '.'

INPUT_DIR = KAGGLE_PREPROC if os.path.exists(KAGGLE_PREPROC) else LOCAL_PREPROC
ML_DIR    = f'{INPUT_DIR}/outputs/for_ml'

print(f'전처리 데이터 경로: {INPUT_DIR}')
print(f'ML 데이터 경로:     {ML_DIR}')

In [ ]:
# 01_preprocessing.ipynb 출력 파일 로드
X_train_hand  = pd.read_parquet(f'{ML_DIR}/X_train_hand.parquet')
X_val_hand    = pd.read_parquet(f'{ML_DIR}/X_val_hand.parquet')
X_test_hand   = pd.read_parquet(f'{ML_DIR}/X_test_hand.parquet')

X_train_tfidf = sp.load_npz(f'{ML_DIR}/X_train_tfidf.npz')
X_val_tfidf   = sp.load_npz(f'{ML_DIR}/X_val_tfidf.npz')
X_test_tfidf  = sp.load_npz(f'{ML_DIR}/X_test_tfidf.npz')

y_train = np.load(f'{ML_DIR}/y_train.npy')
y_val   = np.load(f'{ML_DIR}/y_val.npy')
y_test  = np.load(f'{ML_DIR}/y_test.npy')

print(f'수동 피처 shape: {X_train_hand.shape}')
print(f'TF-IDF shape:   {X_train_tfidf.shape}')
print(f'Train labels:   {y_train.shape}')

In [ ]:
# 01_preprocessing.ipynb 출력 파일 로드
X_train_hand  = pd.read_parquet('outputs/for_ml/X_train_hand.parquet')
X_val_hand    = pd.read_parquet('outputs/for_ml/X_val_hand.parquet')
X_test_hand   = pd.read_parquet('outputs/for_ml/X_test_hand.parquet')

X_train_tfidf = sp.load_npz('outputs/for_ml/X_train_tfidf.npz')
X_val_tfidf   = sp.load_npz('outputs/for_ml/X_val_tfidf.npz')
X_test_tfidf  = sp.load_npz('outputs/for_ml/X_test_tfidf.npz')

y_train = np.load('outputs/for_ml/y_train.npy')
y_val   = np.load('outputs/for_ml/y_val.npy')
y_test  = np.load('outputs/for_ml/y_test.npy')

print(f'수동 피처 shape: {X_train_hand.shape}')
print(f'TF-IDF shape:   {X_train_tfidf.shape}')
print(f'Train labels:   {y_train.shape}')

In [ ]:
# MultinomialNB는 음수 불가 → MaxAbsScaler로 [0,1] 범위로 스케일
scaler = MaxAbsScaler()
X_train_tfidf_scaled = scaler.fit_transform(X_train_tfidf)
X_val_tfidf_scaled   = scaler.transform(X_val_tfidf)
X_test_tfidf_scaled  = scaler.transform(X_test_tfidf)

# 수동 피처 numpy 변환
X_train_h = X_train_hand.values
X_val_h   = X_val_hand.values
X_test_h  = X_test_hand.values

print('데이터 준비 완료')

## 1. 모델 학습 & 평가 유틸리티

In [ ]:
results = []

def evaluate_model(name, model, X_tr, y_tr, X_val, y_val, X_te, y_te):
    """학습 → val/test 평가 → 결과 딕셔너리 반환"""
    # 학습
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0

    # 추론 시간 (test set)
    t0 = time.time()
    y_pred = model.predict(X_te)
    infer_time_ms = (time.time() - t0) / len(y_te) * 1000  # ms per sample

    # 확률 예측 (log loss용)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_te)[:, 1]
    else:
        y_prob = model.decision_function(X_te)
        y_prob = (y_prob - y_prob.min()) / (y_prob.max() - y_prob.min())

    # 모델 파라미터 수 (근사)
    try:
        param_count = sum(p.size for p in [model.coef_] if hasattr(model, 'coef_'))
    except Exception:
        param_count = -1

    # 모델 저장
    with open(f'outputs/models/{name.replace(" ", "_")}.pkl', 'wb') as f:
        pickle.dump(model, f)
    model_size_kb = os.path.getsize(f'outputs/models/{name.replace(" ", "_")}.pkl') / 1024

    # val 메트릭
    y_val_pred = model.predict(X_val)
    val_acc    = accuracy_score(y_val, y_val_pred)
    val_f1     = f1_score(y_val, y_val_pred)

    result = {
        'Model':          name,
        'Val Acc':        round(val_acc, 4),
        'Val F1':         round(val_f1, 4),
        'Test Acc':       round(accuracy_score(y_te, y_pred), 4),
        'Test F1':        round(f1_score(y_te, y_pred), 4),
        'Log Loss':       round(log_loss(y_te, y_prob), 4),
        'Train Time (s)': round(train_time, 2),
        'Infer (ms/sample)': round(infer_time_ms, 4),
        'Model Size (KB)': round(model_size_kb, 1),
    }
    results.append(result)
    print(f'[{name}]  Val Acc={val_acc:.4f}  Test Acc={result["Test Acc"]:.4f}  '
          f'F1={result["Test F1"]:.4f}  LogLoss={result["Log Loss"]:.4f}')
    return model

## 2. Logistic Regression (수동 피처)

In [ ]:
lr_hand = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
evaluate_model(
    'LR (Hand Features)',
    lr_hand,
    X_train_h, y_train,
    X_val_h,   y_val,
    X_test_h,  y_test
)

## 3. Logistic Regression (TF-IDF)

In [ ]:
lr_tfidf = LogisticRegression(C=1.0, max_iter=1000, solver='saga', random_state=42)
evaluate_model(
    'LR (TF-IDF)',
    lr_tfidf,
    X_train_tfidf, y_train,
    X_val_tfidf,   y_val,
    X_test_tfidf,  y_test
)

## 4. Linear SVM (TF-IDF)

In [ ]:
# LinearSVC는 predict_proba 없음 → CalibratedClassifierCV로 래핑
svm_base = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm = CalibratedClassifierCV(svm_base, cv=3)
evaluate_model(
    'Linear SVM (TF-IDF)',
    svm,
    X_train_tfidf, y_train,
    X_val_tfidf,   y_val,
    X_test_tfidf,  y_test
)

## 5. Naive Bayes (TF-IDF)

In [ ]:
nb = MultinomialNB(alpha=0.1)
evaluate_model(
    'Naive Bayes (TF-IDF)',
    nb,
    X_train_tfidf_scaled, y_train,
    X_val_tfidf_scaled,   y_val,
    X_test_tfidf_scaled,  y_test
)

## 6. Decision Tree (수동 피처)

In [ ]:
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
evaluate_model(
    'Decision Tree (Hand)',
    dt,
    X_train_h, y_train,
    X_val_h,   y_val,
    X_test_h,  y_test
)

## 7. Random Forest (수동 피처)

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=15, n_jobs=-1, random_state=42)
evaluate_model(
    'Random Forest (Hand)',
    rf,
    X_train_h, y_train,
    X_val_h,   y_val,
    X_test_h,  y_test
)

## 8. Gradient Boosting (수동 피처)

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1,
    max_depth=5, subsample=0.8, random_state=42
)
evaluate_model(
    'Gradient Boosting (Hand)',
    gb,
    X_train_h, y_train,
    X_val_h,   y_val,
    X_test_h,  y_test
)

## 9. 결과 비교 & 저장

In [ ]:
results_df = pd.DataFrame(results).set_index('Model')
results_df.to_csv('outputs/results/traditional_ml_results.csv')
print(results_df.to_string())

In [ ]:
# 피처 중요도 (Random Forest 기준)
feature_names = list(X_train_hand.columns)
importances   = rf.feature_importances_
feat_imp_df   = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_imp_df   = feat_imp_df.sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=feat_imp_df, x='importance', y='feature', color='steelblue')
plt.title('Feature Importance (Random Forest)\n베이스라인 10개 피처')
plt.tight_layout()
plt.savefig('outputs/results/feature_importance.png', dpi=120)
plt.show()

print('\n최종 결과 저장 완료')
print('  outputs/results/traditional_ml_results.csv')
print('  outputs/results/traditional_ml_comparison.png')
print('  outputs/results/feature_importance.png')

In [ ]:
# 피처 중요도 (Random Forest 기준)
feature_names = list(X_train_hand.columns)
importances   = rf.feature_importances_
feat_imp_df   = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_imp_df   = feat_imp_df.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp_df, x='importance', y='feature', color='steelblue')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()
plt.savefig('outputs/results/feature_importance.png', dpi=120)
plt.show()

print('\n최종 결과 저장 완료')
print('  outputs/results/traditional_ml_results.csv')
print('  outputs/results/traditional_ml_comparison.png')
print('  outputs/results/feature_importance.png')